# Train the power-line fault detector (YOLOv8)
Runtime > Change runtime type > **T4 GPU**. Run cells top to bottom.
You need a **labelled dataset** (YOLO format). Use Option A (Roboflow Universe) or Option B (your own zip).

In [ ]:
!pip -q install ultralytics roboflow
import torch; print('GPU:', torch.cuda.is_available())

## Option A: download a dataset from Roboflow Universe
1. Go to universe.roboflow.com, search e.g. *insulator defect*, *power line*, *transmission tower*, *broken wire*. Check the licence.
2. Open a dataset > **Download Dataset** > Format **YOLOv8** > *show download code*, then paste the three values below.
3. Get a free API key from your Roboflow account settings.

In [ ]:
from roboflow import Roboflow
API_KEY = 'PASTE_YOUR_KEY'
WORKSPACE = 'PASTE_WORKSPACE'
PROJECT = 'PASTE_PROJECT'
VERSION = 1
rf = Roboflow(api_key=API_KEY)
ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8')
DATA_YAML = ds.location + '/data.yaml'
print(DATA_YAML)

## Option B: your own labelled data
Zip a folder with `train/`, `valid/` (each with `images/` and `labels/`) and `data.yaml`, upload it in the Files panel as `data.zip`, then run:

In [ ]:
# !unzip -q data.zip -d mydata
# DATA_YAML = '/content/mydata/data.yaml'

## Train and validate

In [ ]:
from ultralytics import YOLO
m = YOLO('yolov8n.pt')
m.train(data=DATA_YAML, epochs=60, imgsz=640, batch=16)
metrics = m.val()
print('mAP50 on YOUR validation set:', round(float(metrics.box.map50), 3))
print('Class names (must match detector_yolo.py CLASS_MAP, unknown names still work as generic defects):', m.names)

## Download the weights
Put `best.pt` into your project's `models/` folder (rename if needed), run `pip install ultralytics` in your venv, restart the server.

In [ ]:
import shutil
from google.colab import files
shutil.copy(m.trainer.best, 'best.pt'); files.download('best.pt')